# Packet P-033 — Feature Stability Under Bootstrap Resampling

**Decision question:** Are the top-K important features stable across bootstrap resamples, or does the feature ranking shuffle significantly?

**Method:** Meinshausen-Buhlmann stability selection — draw B=100 bootstrap resamples, fit ET, extract permutation importance, compute selection probability for each feature appearing in top-K.

**Fastest falsifier:** If the top-3 set changes composition in >60% of 50 resamples, stability is falsified.

**Success:** At least 3 features have selection probability > 0.6 at K=5.
**Kill:** No feature exceeds selection probability 0.5 at K=5.

In [1]:
"""Cell 1 — Load data, setup permutation importance bootstrap."""
import pandas as pd
import numpy as np
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('perovskite_stability_clean.csv')
FEATURES = [
    'Perovskite_band_gap', 'Pb', 'Sn', 'I', 'Br', 'Cl',
    'MA', 'FA', 'Cs',
    'first_Prvskt_annealing_temperature', 'first_Prvskt_thermal_annealing_time',
    'Perovskite_thickness', 'Cell_area_measured',
    'JV_default_Voc', 'JV_default_Jsc', 'JV_default_FF'
]
TARGET = 'Stability_PCE_T80'
ET_PARAMS = dict(n_estimators=700, max_features='sqrt', min_samples_split=3,
                 min_samples_leaf=1, bootstrap=False, random_state=42)

X = df[FEATURES].fillna(0)
y = np.log1p(df[TARGET])
print(f"Dataset: {len(X)} devices, {len(FEATURES)} features")

Dataset: 1543 devices, 16 features


In [2]:
"""Cell 2 — Bootstrap stability selection (B=100 resamples)."""

B = 100  # bootstrap resamples
Ks = [3, 5, 8]

# Track how often each feature appears in top-K
selection_counts = {K: np.zeros(len(FEATURES)) for K in Ks}

for b in range(B):
    rng = np.random.RandomState(b)
    # Bootstrap resample (with replacement)
    idx = rng.choice(len(X), size=len(X), replace=True)
    X_boot = X.iloc[idx]
    y_boot = y.iloc[idx]
    
    # OOB indices for evaluation
    oob_idx = list(set(range(len(X))) - set(idx))
    if len(oob_idx) < 20:
        continue
    X_oob = X.iloc[oob_idx]
    y_oob = y.iloc[oob_idx]
    
    # Fit model
    params = ET_PARAMS.copy()
    params['random_state'] = b
    model = ExtraTreesRegressor(**params)
    model.fit(X_boot, y_boot)
    
    # Permutation importance on OOB data
    perm = permutation_importance(model, X_oob, y_oob, n_repeats=5,
                                   random_state=b, scoring='r2')
    importances = perm.importances_mean
    
    # Track top-K selections
    ranked_idx = np.argsort(importances)[::-1]
    for K in Ks:
        top_k = ranked_idx[:K]
        selection_counts[K][top_k] += 1
    
    if (b + 1) % 25 == 0:
        print(f"  Completed {b+1}/{B} resamples")

# Compute selection probabilities
print(f"\n{'=' * 70}")
print("FEATURE SELECTION PROBABILITIES (Stability Selection)")
print(f"{'=' * 70}")
print(f"{'Feature':<45} {'Top-3':>8} {'Top-5':>8} {'Top-8':>8}")
print("-" * 70)

stability_rows = []
for i, feat in enumerate(FEATURES):
    probs = {K: selection_counts[K][i] / B for K in Ks}
    print(f"{feat:<45} {probs[3]:>8.2f} {probs[5]:>8.2f} {probs[8]:>8.2f}")
    stability_rows.append({
        'feature': feat,
        'prob_top3': probs[3], 'prob_top5': probs[5], 'prob_top8': probs[8]
    })

stab_df = pd.DataFrame(stability_rows).sort_values('prob_top5', ascending=False)
print(f"\n── Top-5 ranked by selection probability at K=5 ──")
for _, row in stab_df.head(8).iterrows():
    marker = "★" if row['prob_top5'] >= 0.6 else " "
    print(f"  {marker} {row['feature']:<40} {row['prob_top5']:.2f}")

  Completed 25/100 resamples


  Completed 50/100 resamples


  Completed 75/100 resamples


  Completed 100/100 resamples

FEATURE SELECTION PROBABILITIES (Stability Selection)
Feature                                          Top-3    Top-5    Top-8
----------------------------------------------------------------------
Perovskite_band_gap                               0.96     1.00     1.00
Pb                                                0.00     0.01     0.20
Sn                                                0.02     0.05     0.48
I                                                 0.00     0.00     0.00
Br                                                0.00     0.00     0.16
Cl                                                0.00     0.02     0.23
MA                                                0.00     0.00     0.04
FA                                                0.00     0.04     0.26
Cs                                                0.00     0.05     0.42
first_Prvskt_annealing_temperature                0.00     0.48     0.92
first_Prvskt_thermal_annealing_time      

In [3]:
"""Cell 3 — Evaluate thresholds and save."""

# Success: at least 3 features have selection probability > 0.6 at K=5
# Kill: no feature exceeds selection probability 0.5 at K=5

stable_at_06 = stab_df[stab_df['prob_top5'] >= 0.6]
stable_at_05 = stab_df[stab_df['prob_top5'] >= 0.5]

print(f"Features with selection probability >= 0.6 at K=5: {len(stable_at_06)}")
for _, row in stable_at_06.iterrows():
    print(f"  {row['feature']}: {row['prob_top5']:.2f}")

print(f"\nFeatures with selection probability >= 0.5 at K=5: {len(stable_at_05)}")

# Compare with P-020 consensus features
p020_consensus = ['Perovskite_band_gap', 'JV_default_Jsc', 'Cell_area_measured',
                   'JV_default_Voc', 'JV_default_FF', 'Perovskite_thickness']
bootstrap_top5 = stab_df.head(5)['feature'].tolist()
overlap = set(p020_consensus[:5]) & set(bootstrap_top5)
print(f"\nOverlap with P-020 consensus top-5: {len(overlap)}/5")
print(f"  P-020: {p020_consensus[:5]}")
print(f"  Bootstrap: {bootstrap_top5}")

# Status
if len(stable_at_06) >= 3:
    status = "Confirmed"
elif len(stable_at_05) > 0:
    status = "Promising"
else:
    status = "Negative"

# Save
stab_df.to_csv('Packet_P033_Feature_Bootstrap_Stability.csv', index=False)
print(f"\nSaved: Packet_P033_Feature_Bootstrap_Stability.csv")

print(f"\n{'=' * 60}")
print(f"P-033 STATUS: {status}")
print(f"{'=' * 60}")
if status == "Confirmed":
    print("Feature rankings are stable under bootstrap resampling.")
elif status == "Promising":
    print("Some features are moderately stable, but top-K set is noisy.")
else:
    print("No reliably important features — rankings shuffle across resamples.")

Features with selection probability >= 0.6 at K=5: 4
  Perovskite_band_gap: 1.00
  Perovskite_thickness: 0.99
  Cell_area_measured: 0.99
  first_Prvskt_thermal_annealing_time: 0.83

Features with selection probability >= 0.5 at K=5: 4

Overlap with P-020 consensus top-5: 2/5
  P-020: ['Perovskite_band_gap', 'JV_default_Jsc', 'Cell_area_measured', 'JV_default_Voc', 'JV_default_FF']
  Bootstrap: ['Perovskite_band_gap', 'Perovskite_thickness', 'Cell_area_measured', 'first_Prvskt_thermal_annealing_time', 'first_Prvskt_annealing_temperature']

Saved: Packet_P033_Feature_Bootstrap_Stability.csv

P-033 STATUS: Confirmed
Feature rankings are stable under bootstrap resampling.
